In [ ]:
from IPython.display import display, HTML

display(HTML("""
<style>
/* JupyterLab: center notebook and leave margins */
.jp-NotebookPanel-notebook {
  width: 85% !important;              
  margin-left: auto !important;
  margin-right: auto !important;
  max-width: 1100px !important;       /* optional cap */
}

/* Make output area match nicely */
.jp-OutputArea {
  max-width: 100% !important;
}
</style>
"""))
# make high-resolution of images.
%config InlineBackend.figure_format = 'svg'
print("setup jnk layout and figure format.")

<a id="0"></a> <br>
## Table of Contents
- [0. Preparing Your Environment](#00)
  - [0.1 Install Anaconda and setup environment](#01)
  - [0.2 Checking your device](#02)
- [1. Explore LLMs via ollama](#10)
  - [1.1 Install ollama and download LLMs](#11)
  - [1.2 Get response for a given prompt](#12)
  - [1.3 Get response probability from LLMs](#13)
- [2. Basics for Python and spaCy](#20)
  - [2.1 Unicode for text data](#21)
  - [2.2 Python basics: regular expression](#22)
  - [2.3 (Old way) Tokenization tool: spaCy](#23)
- [3. Datasets Exploration](#30)
  - [3.1 Heap's Law](#31)
  - [3.2 Explore wikitext-2 dataset](#32)
  - [3.3 Explore wikitext-103 dataset](#33)
  - [3.4 BookCorpus datasets](#34)
- [4. LLMs tokenization (Modern Way)](#40)
  - [4.1 Subword tokenization: BPE](#41)
  - [4.2 Subword tokenization: WordPiece](#42)
  - [4.3 Subword tokenization: Unigram](#43)
- [5. Minimum Edit Distance](#50)
  - [5.1 Algorithm analysis](#51)
  - [5.2 Algorithm implementation](#52)
- [6. Submit Your Work](#60)
- [7. Useful Commands and References](#70)

<a id="00"></a> 
## 0. Preparing Your Environment

<a id="01"></a> 
### 0.1 Install Anaconda and setup environment

<div style="
  background:#f6f6f6;
  border-left:6px solid #999;
  border-radius:8px;
  padding:14px 18px;
  max-width:900px;
  margin:16px auto;
  text-align:center;
  font-size:16px;
  line-height:1.5;
">
    If you are using macOS or Linux, you are ready to go. 
  <b>⚠️ If you are using Windows, I strongly recommend installing Ubuntu 22.04 via
  <a href="https://ubuntu.com/desktop/wsl" target="_blank" rel="noopener">WSL</a>.
  In the rest of the instructions, I assume you are using Ubuntu/Linux or macOS.</b>
</div>

- **Step 1: Install Anaconda.** Download and install Anaconda from: [https://www.anaconda.com/download](https://www.anaconda.com/download).

---

- **Step 2: Setup your git.** Make sure you have [git](https://git-scm.com/install/) and ssh in your system (macOS usually already has both git and ssh). After the above step, you are ready to download our course github project. In your Ubuntu/Linux system, install git and sssh via

  ```sudo apt update```

  ```sudo apt install -y git openssh-client```
  
  Then, generate an SSH key (ed25519)

  ```ssh-keygen -t ed25519 -C "your_email@example.com"```
  
  Press Enter for the default location, and optionally set a passphrase. This creates:

  - ~/.ssh/id_ed25519 (private key)
  - ~/.ssh/id_ed25519.pub (public key)
    
  Then, you can start ssh-agent and add the key:

  ```eval "$(ssh-agent -s)"```


  ```ssh-add ~/.ssh/id_ed25519```


  Add the public key to GitHub, that is, copy the generated key:

  ```cat ~/.ssh/id_ed25519.pub```

  It will be something like, ```ssh-ed25519 XXXXXXX...XXXXXXX your-email@xxx.com```. **Copy this line**, and then in GitHub: **Settings → SSH and GPG keys → New SSH key → paste.** To test the SSH connection, you can use ```ssh -T git@github.com```, if it says something like “Hi USERNAME! …”, you’re good. After these steps, you are ready to clone our course github via
  
  ```git clone git@github.com:baojian/llm-26.git```

  After the above steps, you are ready to create our course env.

---


- **Step 3: Create and activate the course environment.** Make sure you have installed the conda environment.

  - **Option 1:** If you have GPU in your machine (Linux + NVIDIA GPU), please create your env via
 
    ```cd llm-26/lecture-01-tokenization/ # make sure your are in our course folder.```
 
    ```conda env create -f environment-gpu.yml```

    Activate it with:

    ```conda activate llm-26-gpu```

    Note: environment-gpu.yml uses ```pytorch-cuda=12.1```. If your GPU driver/CUDA setup does not support CUDA 12.1, change this version accordingly (e.g., 11.8).

  - **Option 2:** If you use macOS / Windows (WSL) / Linux CPU-only, please create your env via
 
    ```conda env create -f environment.yml```
 
    Activate it with:
 
    ```conda activate llm-26-cpu```
 
    Note, if you got errors when installing ```en_core_web_sm``` or ```zh_core_web_sm```, please remove these two from yml file and install separately in your env via
 
    ```shell
    conda activate llm-26 # or conda activate llm-26-gpu
    python -m spacy download en_core_web_sm
    python -m spacy download zh_core_web_sm
    ```
    After the above steps, you are ready to download our course github project.

---

- **Step 4: Open your jupyter notebook.**

  All your code will run on Jupyter notebook, you can activate jupyter notebook, via

  ```cd llm-26 # make sure you are in our course folder```

  ```conda activate llm-26-cpu # or conda activate llm-26-gpu```
  
  ```jupyter lab```

  For students using Windows WSL (Ubuntu 22.04), even though Jupyter runs inside WSL (Ubuntu), you can open it directly in the Windows browser via port forwarding (usually automatic). In WSL Ubuntu, start Jupyter like this:

  ```jupyter lab --no-browser --ip=127.0.0.1 --port=8888```

  It will print a URL like: ```http://127.0.0.1:8888/lab?token=...```

  Now on Windows, open your browser and go to: ```http://localhost:8888/lab``` Paste the token if it asks. On WSL2, Windows automatically forwards localhost:8888 to the WSL instance in most setups. If localhost:8888 doesn’t work, in WSL run: ```hostname -I```, suppose it prints something like 172.27.123.45 ..., then open in Windows: ```http://172.27.123.45:8888/lab```

- **Load all necessary packages**

  At this point, your env is ready to use. We import all necessary packages here.

In [ ]:
import re
import os
import time
import math

import shutil
import socket
import platform
import requests
import datetime
import tarfile

import spacy
import torch
import numpy as np

from transformers import AutoTokenizer
from transformers import Qwen2Tokenizer, Qwen2TokenizerFast

import tiktoken
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents
from tokenizers import decoders

from transformers import AutoTokenizer
from tokenizers import Tokenizer
from tokenizers import normalizers
from tokenizers.models import WordPiece
from tokenizers.normalizers import NFD, Lowercase, StripAccents
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

from datasets import load_dataset
import matplotlib.pyplot as plt

jnk_start_time = time.time()

<a id="02"></a> 
### 0.2 Checking your device

- After install conda and your env, you can check whether these packages are installed in the right way.
```shell
python -c "import torch; print('torch', torch.__version__); print('cuda?', torch.cuda.is_available()); print('mps?', getattr(torch.backends,'mps',None) is not None and torch.backends.mps.is_available()); print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu'); print('using:', 'cuda' if torch.cuda.is_available() else ('mps' if (getattr(torch.backends,'mps',None) is not None and torch.backends.mps.is_available()) else 'cpu'))"
```

In [ ]:
!python -c "import torch; print('torch', torch.__version__); print('cuda?', torch.cuda.is_available()); print('mps?', getattr(torch.backends,'mps',None) is not None and torch.backends.mps.is_available()); print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu'); print('using:', 'cuda' if torch.cuda.is_available() else ('mps' if (getattr(torch.backends,'mps',None) is not None and torch.backends.mps.is_available()) else 'cpu'))"

- The following code provides a more symmetric way to perform the same checks. In our lectures, some datasets are very large, so it’s a good idea to make sure you have enough disk space. You can check your disk, memory, and CPU information using the code below.

In [ ]:
def detect_torch_device(verbose: bool = True) -> str:
    """
    Returns one of: 'cuda', 'mps', 'cpu'
    Priority: CUDA GPU > Apple MPS > CPU
    """
    has_cuda = torch.cuda.is_available()
    has_mps = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()

    if has_cuda:
        device = "cuda"
    elif has_mps:
        device = "mps"
    else:
        device = "cpu"

    if verbose:
        print(f"torch: {torch.__version__}")
        print(f"device: {device}")

        if has_cuda:
            print(f"cuda devices: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  [{i}] {torch.cuda.get_device_name(i)}")
        elif has_mps:
            print("mps available: True (Apple Metal)")
        else:
            print("cpu only")

    return device

device = detect_torch_device()
# generate 2x3 random matrix to check torch
x = torch.rand(2, 3)
x = x.to(device)
print("device:", x.device)

- **Exercise System Info Cell**

In [ ]:
def bytes_to_gb(x: int) -> float:
    return x / (1024 ** 3)

def system_report(path: str = "."):
    print("=== System Report ===")
    print("OS:", platform.system(), platform.release())
    print("Platform:", platform.platform())
    now = datetime.datetime.now().astimezone()
    print("Local time:", now.strftime("%Y-%m-%d %H:%M:%S %Z%z"))
    print("Hostname  :", socket.gethostname())
    print("User      :", os.getenv("USER") or os.getenv("USERNAME") or "unknown")
    
    print("\n=== OS / Python ===")
    print("OS        :", platform.system(), platform.release())
    print("Version   :", platform.version())
    print("Machine   :", platform.machine())
    print("Processor :", platform.processor() or "unknown")
    print("Python    :", platform.python_version())

    # CPU
    print("\n--- CPU ---")
    print("CPU cores (logical):", os.cpu_count())

    # Memory (best-effort, cross-platform)
    print("\n--- Memory (RAM) ---")
    try:
        import psutil  # you already have this in env
        vm = psutil.virtual_memory()
        print(f"Total: {bytes_to_gb(vm.total):.2f} GB")
        print(f"Available: {bytes_to_gb(vm.available):.2f} GB")
        print(f"Used: {bytes_to_gb(vm.used):.2f} GB ({vm.percent}%)")
    except Exception as e:
        print("psutil not available or failed:", e)

    # Disk
    print("\n--- Disk ---")
    total, used, free = shutil.disk_usage(path)
    print("Path checked:", os.path.abspath(path))
    print(f"Total: {bytes_to_gb(total):.2f} GB")
    print(f"Free:  {bytes_to_gb(free):.2f} GB")
    print(f"Used:  {bytes_to_gb(used):.2f} GB")

    # PyTorch device
    print("\n--- PyTorch Device ---")
    try:
        import torch
        cuda = torch.cuda.is_available()
        mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
        print("torch:", torch.__version__)
        print("CUDA available:", cuda)
        print("MPS available:", mps)
        if cuda:
            print("GPU:", torch.cuda.get_device_name(0))
        device = "cuda" if cuda else ("mps" if mps else "cpu")
        print("Suggested device:", device)
    except Exception as e:
        print("torch not available or failed:", e)

system_report(".")

<a id="10"></a> 
## 1. Explore LLMs via ollama

In this section, we’ll have some fun exploring LLMs with [ollama](https://github.com/ollama/ollama). 

<a id="11"></a> 
### 1.1  Install ollama and download LLMs

We can download some popular LLMs such as Qwen series.


- **Step 1:** Please download and install ollama via [download](https://ollama.com/download).

- **Step 2:** After the installation of ollama, you can open a terminal and download qwen3:0.6b and qwen3:1.7b via:

  ```ollama run qwen3:0.6b```

  It will automatically download the model into your local disk. It takes about 498MB disk space. Similarly, you can run a larger version ```qwen3:1.7b``` and it takes about 1.3GB disk space. Let us test 0.6b model

  ```ollama run qwen3:0.6b```

- **Step 2:** Ask a question, like: ```hi, where is Fudan located at?```, it gives us:

<div style="text-align:center;">
<img src="assets/ollama-terminal.png" alt="ollama terminal" style="width:75%;">
</div>

<a id="12"></a> 
### 1.2 Get response for a given prompt

- **Call Ollama API**

  Next, we go a little deeper: we will call the Ollama API from Python to generate outputs given prompts. You can check whether Ollama is running by typing [http://127.0.0.1:11434/](http://127.0.0.1:11434/) in Chrome.

In [ ]:
# Make sure bypass proxies for local services (e.g., Ollama on localhost:11434)
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

s = requests.Session()
# IMPORTANT <- ignore ALL proxy env vars
s.trust_env = False  
# make sure ollama is running
print(s.get("http://127.0.0.1:11434/api/version", timeout=3).json())

# !!!! make sure to import after the s.trust_env !!!!

import ollama
from ollama import chat
from ollama import ChatResponse

s = requests.Session()
s.trust_env = False  # <- ignore ALL proxy env vars

response: ChatResponse = chat(model='qwen3:0.6b', messages=[
  {
    'role': 'user',
    'content': 'Fudan University is located in which city? Answer with one word.',
  },
])
print(response['message']['content'])

- **Will the same prompt always produce the same output?**

In [ ]:
models = ["qwen3:0.6b", "qwen3:1.7b"]
prompt = "Fudan University is located in which city? Answer with one word."

for model in models:
    print('-' * 50)
    start_time = time.time()
    for _ in range(10):
        resp = ollama.generate(
            model = model,
            prompt = prompt
        )
        print(f"{model} with resp {_ + 1}: {resp["response"]}")
    print(f'total runtime of 10 responses of {model} is: {time.time() - start_time:.1f} seconds')

- **Some key observations**:
  - The smaller model, **Qwen3:0.6B**, tends to produce lower-quality answers, while the larger model, **Qwen3:1.7B**, generally produces higher-quality answers.
  - The **responses are somewhat random**: running the same prompt multiple times may yield different outputs. There are ways to make the output deterministic. For example, the code below sets decoding options so that the model always selects the highest-probability token at each step.
    <div style="background:#f5f5f5; border:1px solid #ddd; border-radius:8px; max-width:900px; margin:12px auto;">
        <pre style="margin:0;"><code class="language-python">
            resp = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "temperature": 0.0,
                "top_p": 1.0,
                "top_k": 0,
                # optional:
                "num_predict": 32,
            },
            )
            print(resp["response"])</code></pre>
    </div>
    
  Let us generate some response that are not one word but a sequence of words.

- **Get a paragraph of response with qwen3:1.7b**

In [ ]:
model = "qwen3:1.7b"
prompt = "I am an undergraduate student, please explain LLMs in three sentences."
resp = ollama.generate(
            model=model,
            prompt=prompt
        )
print(f"Prompt: {prompt} \nResp: {resp["response"]}")

<a id="13"></a> 
### 1.3 Get response probability from LLMs

- **Get token probability (Observe: single word could be tokenized into subwords.)**

In [ ]:
model = "qwen3:0.6b" # "qwen3:1.7b"
prompt = "Fudan University is located in which city? Answer with one word."
num_top_tokens = 20 # number of top alternatives per generated token
resp = ollama.generate(
    model = model,
    prompt = prompt,
    stream = False,
    logprobs = True,
    think = False,
    top_logprobs = num_top_tokens
)
print("response:", repr(resp["response"]))

# Each element usually corresponds to one generated token
for i, lp in enumerate(resp.get("logprobs", [])):
    tok = lp.get("token")
    logp = lp.get("logprob")
    p = math.exp(logp) if logp is not None else None
    print(f"{i:02d} token={tok!r:>16} logp={logp: .4f} p={p:.4f}")

- **Get specific token probability distribution via logprobs (without thinking)**

In [ ]:
model = "qwen3:0.6b"
prompt = "Fudan University is located in which city? Answer with one word."

res = ollama.generate(
    model=model,
    prompt=prompt,
    logprobs=True,
    think = False,
    top_logprobs=10,
    options={"temperature": 0.0, # greedy decoding, it pick the maximal token
             "num_predict": 20,
            "think": False # do not use thinking model.
            },
)

answer = ''
lp = res["logprobs"]
tokens = [d.get("token", "") for d in lp]
print(f'We use model: {model}')
for i in range(len(lp)):
    tok = lp[i].get("token", "")
    logp = lp[i].get("logprob", None)
    alts = lp[i].get("top_logprobs", [])
    p = math.exp(logp) if logp is not None else float("nan")
    if tok == "\n" or tok == "\n\n": # stop when answer ends (often newline).
        break
    answer += tok
    print(f"--- top probabilities of token-{i:02d} ---")
    for a in alts[:20]:
        prob_a = math.exp(a['logprob'])
        print(f"{a['token']!r:>12}:{prob_a:.5f}")
    print(f"\nPartial Response: {answer}\n")
print(f"Final Response: {answer}")

- **LLMs are probabilitic distributions**

From the outputs above, you can see that the response from these LLMs may differ from run to run. In fact, LLMs are **probabilistic models**: given the same prompt (i.e., question), they may produce different responses (i.e., answers). We can describe this inference process using mathematical notation. Let $p_\theta(\cdot)$ denote a trained language model. Here, you can think of $p_\theta$ as **Qwen3:0.6B**, **Qwen3:1.7B**, or any other model. Given the following prompt

<blockquote style="
  background:#e8f4ff;
  border-left:4px solid #7db7ff;
  padding:10px 14px;
  border-radius:6px;
  margin: 10px 10%;">
Prompt = Fudan University is located in which city? Answer with one word.
</blockquote>

**This prompt is a sequence of tokens**. The model outputs a response, which is also a sequence of tokens. In other words, the model uses an inference procedure to predict the next token(s) conditioned on the prompt. In math, it models the conditional probability
<blockquote style="
  background:#e8f4ff;
  border-left:4px solid #7db7ff;
  padding:10px 14px;
  border-radius:6px;
  margin: 10px 10%;">$$p_\theta \left(w_{n+1} \mid w_1,w_2,\ldots,w_n\right),$$</blockquote>
where

- **Prompt**$[w_1,w_2,\ldots,w_n]$ =[ Fudan University is located in which city? Answer with one word. ],
- **Response** = $[w_{n+1}]$ (in this simplified one-word setting).

The model then outputs the most likely token $w_{n+1}$ (or samples from the distribution) using its own inference algorithm. We will discuss inference strategies in later lectures. By the definition of conditional probability, we have

<blockquote style="
  background:#e8f4ff;
  border-left:4px solid #7db7ff;
  padding:10px 14px;
  border-radius:6px;
  margin: 10px 10%;">
$$
p_\theta \left(w_{n+1} \mid w_1,w_2,\ldots,w_n\right) =
\frac{
p_\theta \left(w_1,w_2,\ldots,w_n,w_{n+1}\right)
}{
p_\theta \left(w_1,w_2,\ldots,w_n\right)
}.
$$
</blockquote>

The key takeaway is: **if we can learn a model that assigns probabilities to sequences of arbitrary length**, i.e., $p_\theta(w_1,w_2,\ldots,w_k)$ for any positive integer $k$, then conditional probabilities follow naturally from these joint probabilities. Let us do one more example to see when the response is more than one word.

- **Get a sequence of token probabilities from Qwen3:1.7b**

In [ ]:
model = "qwen3:1.7b"
prompt = "Fudan University is located in which city? Answer with one word."

res = ollama.generate(
    model=model,
    prompt=prompt,
    logprobs=True,
    think = False,
    top_logprobs=10,
    options={"temperature": 0.0, # greedy decoding, it pick the maximal token
             "num_predict": 20,
            "think": False # do not use thinking model.
            },
)
print(res["response"])

- If we do not use thinking mode, it gives the above response. However, we can still see how the token *Shanghai* is chosen during the inference stage. You will see these subword tokens have high probabilities.

In [ ]:
model = "qwen3:1.7b"
prompt = "Fudan University is located in which city? Answer with one word."
print(f'We use model: {model}')

res = ollama.generate(
    model=model,
    prompt=prompt,
    logprobs=True,
    think = False, # Do not use the thinking model.
    top_logprobs=10,
    options={"temperature": 0.0, # greedy decoding, it pick the maximal token
             "num_predict": 20,
            "think": False # do not use thinking model.
            },
)

answer = ''
lp = res["logprobs"]
tokens = [d.get("token", "") for d in lp]
for i in range(len(lp)):
    tok = lp[i].get("token", "")
    logp = lp[i].get("logprob", None)
    alts = lp[i].get("top_logprobs", [])
    p = math.exp(logp) if logp is not None else float("nan")
    # stop when answer ends (often newline).
    if tok == "\n" or tok == "\n\n": 
        break
    answer += tok
    print(f"--- top probabilities of token-{i:02d} ---")
    for a in alts[:5]:
        prob_a = math.exp(a['logprob'])
        print(f"{a['token']!r:>5}:{prob_a:.5f}", end="")
    print(f"\nPartial Response: {answer}\n")
print(f"Final Response: {answer}")

<div style="
  background:#ffe8e8;
  border:1px solid #ff9a9a;
  border-left:6px solid #ff5c5c;
  padding:12px 14px;
  border-radius:8px;
  margin:12px 10%;
  font-weight:600;">
<center>Above response process is essentially next-token prediction task as qwen3:1.7b is an AR model! </center>
</div>

<a id="20"></a> 
## 2. Basics for Python and spaCy

- **Python**: We will use Python-3.12 in our course.
  
- **spaCy**: As introduced in [https://github.com/explosion/spaCy](https://github.com/explosion/spaCy), [spaCy](https://spacy.io/) is a library for advanced Natural Language Processing in Python and Cython. It's built on the very latest research, and was designed from day one to be used in real products. We will use it to demonostrate how to do text tokenization.

- **nltk tool (outdated, will not cover in our lectures)**: In our previous courses, we usually introduce [https://github.com/nltk/nltk](https://github.com/nltk/nltk) tool for tokenization stuff. But we will not cover this part as these tools are largely iirelvant and outdated.

In [ ]:
# packages needed in this section
import re
import os
import spacy
import numpy as np
import matplotlib.pyplot as plt

<a id="21"></a> 
### 2.1 Unicode for text data

- **Encoding Strings to Bytes**
  
  Use the *.encode()* method on a string object to convert it into a bytes object using UTF-8. 

In [ ]:
my_string = "Hello, World! 你好，世界！"
encoded_bytes = my_string.encode('utf-8') # Encode the string to UTF-8 bytes
print(encoded_bytes)

- **Decoding Bytes to Strings**
  
  Use the *.decode()* method on a bytes object to convert it back into a string, specifying the encoding used to generate the bytes. 

In [ ]:
encoded_bytes = b'Hello, World! \xe4\xbd\xa0\xe5\xa5\xbd\xef\xbc\x8c\xe4\xb8\x96\xe7\x95\x8c\xef\xbc\x81'
decoded_string = encoded_bytes.decode('utf-8') # Decode the bytes back to a string
print(decoded_string)

- **Write and Read utf-8 text into/from files**

In [ ]:
with open('unicode_file.txt', 'w', encoding='utf-8') as f:
    f.write("Some text with a special character: é, 你好，世界！")

with open('unicode_file.txt', 'r', encoding='utf-8') as f:
    content = f.read()
    print(content)

- **ASCII (American Standard Code for Information Interchange)**.

    ASCII represented each character with a single byte in the unicode. UTF-8 is a variable-length encoding format. In Python 3, UTF-8 is the default encoding for source files, strings, and many operations

In [ ]:
import pandas as pd

rows = []
for dec in range(128):
    ch = chr(dec)
    ch_disp = repr(ch)[1:-1] if (dec < 32 or dec == 127) else ch
    rows.append({"Ch": ch_disp, "Hex": f"{dec:02X}", "Dec": dec})
df = pd.DataFrame(rows)

nblocks = 8
chunk = (len(df) + nblocks - 1) // nblocks  # ceil

blocks = []
for i in range(nblocks):
    part = df.iloc[i * chunk : (i + 1) * chunk].reset_index(drop=True)
    part.columns = pd.MultiIndex.from_product([[f"Block {i+1}"], part.columns])
    blocks.append(part)

df3 = pd.concat(blocks, axis=1)
sty = (
    df3.style
      .set_properties(**{"text-align": "center"})
      .set_table_styles([
          {"selector": "th.col_heading.level0", "props": [("text-align", "center")]},
          {"selector": "th, td", "props": [("border", "1px solid #bbb"), ("padding", "2px 6px")]},
          {"selector": "table", "props": [("border-collapse", "collapse")]},
      ])
)
sty

In [ ]:
my_string = "Hello, World! अनुच्छेद, 你好，世界！"

encodings = ["utf-8", "utf-16-le", "gb18030"] # diiferent encoding method
print("\nEncoded bytes (hex):")
for enc in encodings:
    b = my_string.encode(enc)
    print(f"{enc:9s}  {b.hex(' ')}")

# 3) Per-character UTF-8 bytes (useful for teaching)
print("\nPer-Char      Code-Point   UTF-8-Bytes")
for ch in my_string:
    b = ch.encode("utf-8")
    print(f"{repr(ch):>6}    ->    U+{ord(ch):04X}    ->    {b.hex(' ')}")

<a id="22"></a> 
### 2.2 Python basics: regular expression

- **Disjunction []**

In [ ]:
# Task: Find woodchuck or Woodchuck : Disjunction

test_str = "This string contains Woodchuck and woodchuck."

result=re.search(pattern="[wW]oodchuck", string=test_str)
print("Matched" if result is not None else "Not found")
result=re.search(pattern=r"[wW]ooodchuck", string=test_str)
print("Matched" if result is not None else "Not found")

In [ ]:
# Find the word "woodchuck" in the following test string

test_str = "interesting links to woodchucks ! and lemurs!"
result = re.search(pattern="woodchuck", string=test_str)
print("Matched" if result is not None else "Not found")

In [ ]:
# Find !, it follows the same way:

test_str = "interesting links to woodchucks ! and lemurs!"
result = re.search(pattern="!", string=test_str)
print("Matched" if result is not None else "Not found")
result = re.search(pattern="!!", string=test_str)
print("Matched" if result is not None else "Not found")
assert re.search(pattern="!!", string=test_str) == None # match nothing

- **Disjunction any digit [0-9]**

In [ ]:
# Find any single digit in a string.

result=re.search(pattern=r"[0123456789]", string="plenty of 7 to 5")
print("Matched" if result is not None else "Not found", result)
result=re.search(pattern=r"[0-9]", string="plenty of 7 to 5")
print("Matched" if result is not None else "Not found", result)

- **Negation:[^**

In [ ]:
# Negation: If the caret ^ is the first symbol after [,
# the resulting pattern is negated. For example, the pattern 
# [^a] matches any single character (including special characters) except a.

# -- not an upper case letter
print(re.search(pattern=r"[^A-Z]", string="Oyfn pripetchik"))
# -- neither 'S' nor 's'
print(re.search(pattern=r"[^Ss]", string="I have no exquisite reason for't"))
# -- not a period
print(re.search(pattern=r"[^.]", string="our resident Djinn"))
# -- either 'e' or '^'
print(re.search(pattern=r"[e^]", string="look up ^ now"))
# -- the pattern ‘a^b’
print(re.search(pattern=r'a\^b', string=r'look up a^b now'))

- **Or Operation: |**

In [ ]:
# More disjuncations with or or operation
str1 = "Woodchucks is another name for groundhog!"
result = re.search(pattern="groundhog|woodchuck",string=str1)
print(result)

In [ ]:
str1 = "Find all woodchuckk Woodchuck Groundhog groundhogxxx!"
result = re.findall(pattern="[gG]roundhog|[Ww]oodchuck",string=str1)
print(result)

- **Special Operations: ?,*,+,.,$**

In [ ]:
# Some special chars

# ?: Optional previous char
str1 = "Find all color colour colouur colouuur colouyr"
result = re.findall(pattern="colou?r",string=str1)
print(result)

# *: 0 or more of previous char
str1 = "Find all color colour colouur colouuur colouyr"
result = re.findall(pattern="colou*r",string=str1)
print(result)

# +: 1 or more of previous char
str1 = "baa baaa baaaa baaaaa"
result = re.findall(pattern="baa+",string=str1)
print(result)
# .: any char
str1 = "begin begun begun beg3n"
result = re.findall(pattern="beg.n",string=str1)
print(result)
str1 = "The end."
result = re.findall(pattern=r"\.$",string=str1)
print(result)
str1 = "The end? The end. #t"
result = re.findall(pattern=r".$",string=str1)
print(result)

In [ ]:
# find all "the" in a raw text use above operations.

text = "If two sequences in an alignment share a common ancestor, \
mismatches can be interpreted as point mutations and gaps as indels (that \
is, insertion or deletion mutations) introduced in one or both lineages in \
the time since they diverged from one another. In sequence alignments of \
proteins, the degree of similarity between amino acids occupying a \
particular position in the sequence can be interpreted as a rough \
measure of how conserved a particular region or sequence motif is \
among lineages. The absence of substitutions, or the presence of \
only very conservative substitutions (that is, the substitution of \
amino acids whose side chains have similar biochemical properties) in \
a particular region of the sequence, suggest [3] that this region has \
structural or functional importance. Although DNA and RNA nucleotide bases \
are more similar to each other than are amino acids, the conservation of \
base pairs can indicate a similar functional or structural role."
matches = re.findall("[^a-zA-Z][tT]he[^a-zA-Z]", text)
print(matches)

In [ ]:
# Above has spaces before or after each word. A nicer way is to do the following
matches = re.findall(r"\b[tT]he\b", text)
print(matches)

- **Real-world task: match English-like words [A-Za-z]+(?:'[A-Za-z]+)?**

  You can decompose this task into:
  - [A-Za-z]+: Match one or more letters (A–Z or a–z). Examples: cat, PlayStation, Media
  - (?:...): A non-capturing group (grouping for structure, but findall won’t return it separately).
  - [A-Za-z]+: inside the group, Match a literal apostrophe (撇号) ' followed by one or more letters. Examples matched as a whole word: don't, I'm, teacher's
  - ?: after the group, Makes the apostrophe+letters part optional (0 or 1 time).
  - So it matches:
    - ✅ we, Valkyria, PlayStation, don't, teacher's
    - ❌ numbers (2011), hyphenated tokens (real-time), non-ASCII letters (Senjō → only Senj), punctuation, @-@, etc.

In [ ]:
word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

text = """don't Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 ,\
lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles \
III outside Japan , is a tactical role @-@ playing video game developed by Sega and \
Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , \
it is the third game in the Valkyria series . Employing the same fusion of tactical \
and real @-@ time gameplay as its predecessors"""

tokens = word_re.findall(text)
print(len(tokens))
print(tokens)

<a id="23"></a> 
### 2.3 Tokenization using spaCy (Old way)

Usually, there are two type of tokenizations:

- **Top-down tokenization (classic)**: We define a standard and implement rules to implement that kind of tokenization.
  - word tokenization (spaCy, nltk)
  - charater tokenization (also can be done via spaCy)
    
- **Bottom-up tokenization (modern)**: We use simple statistics of letter sequences to break up words into subword tokens.
  - subword tokenization (modern LLMs use this type!)


**Tokenization via spaCy.** We assume you already installed spaCy tool. If you haven't installed yet, open your terminal and activate your llm-26 env, and then run the following to download English and Chinese LMs.
  
  ```python -m spacy download en_core_web_sm```

  ```python -m spacy download zh_core_web_sm```



- **Top-down (rule-based) word tokenization: Tokenization via white space**

In [ ]:
# Use split method via the whitespace " "
text = """While the Unix command sequence just removed all the numbers and punctuation"""
print(text.split(" "))

In [ ]:
# But, we have punctuations, icons, and many other small issues.

text = """Don't you love 🤗 Transformers? We sure do."""
print(text.split(" "))

- **White space cannot tokenize Chinese,Japanese,...**

In [ ]:
text = '姚明进入总决赛'
print(text.split(" "))

- **spaCy works much better: Chinese tokenization**

In [ ]:
from spacy.lang.zh import Chinese
nlp = spacy.load("zh_core_web_sm")

text = '姚明进入总决赛'
doc = nlp(text)
print([token for token in doc])

# ? can be successfully split from Transformers
text = """Don't you love 🤗 Transformers? We sure do."""
doc = nlp(text)
print([token for token in doc]) 

# Chinese Character level tokenization
nlp_ch = Chinese()
text = '姚明进入总决赛'
print([*nlp_ch(text)])

- **spaCy works much better: English tokenization**

In [ ]:
import spacy
from spacy.tokens import Doc

nlp = spacy.blank("en")

text = "Hello, world!"
chars = [c for c in text]
doc = Doc(nlp.vocab, words=chars)

print([t.text for t in doc])

# ignore white space
chars = [c for c in text if not c.isspace()]
doc = Doc(nlp.vocab, words=chars)

print([t.text for t in doc])

- **spaCy handles special characters**

In [ ]:
text = """Special characters and numbers will need to be kept in prices ($45.55) and dates (01/02/06); 
we don’t want to segment that price into separate tokens of “45” and “55”. And there are URLs (https://www.stanford.edu),
Twitter hashtags (#nlproc), or email addresses (someone@cs.colorado.edu)."""
text = text.replace("\n", " ").strip()
nlp = spacy.load('en_core_web_sm')
doc = nlp(text)
all_tokens = [token for token in doc]
print(all_tokens)

- **Sentence-level tokenization via spaCy**

In [ ]:
text = """自然语言处理（英语：Natural Language Processing，缩写作 NLP）是人工智能和语言学领域的交叉学科，\
研究计算机处理、理解与生成人类语言的技术。此领域探讨如何处理及运用自然语言；自然语言处理包括多方面和步骤，\
基本有认知、理解、生成等部分。自然语言认知和理解是让电脑把输入的语言变成结构化符号与语义关系，\
然后根据目的再处理。自然语言生成系统则是把计算机数据转化为自然语言。\
自然语言处理要研制表示语言能力和语言应用的模型, 建立计算框架来实现并完善语言模型，\
并根据语言模型设计各种实用系统及探讨这些系统的评测技术。‌‌"""
nlp = spacy.load("zh_core_web_sm")
doc = nlp(text)
print([sent.text for sent in doc.sents])

- Or, you can spaCy provided sentences function

In [ ]:
 # You need to install it via: python -m spacy download zh_core_web_sm
from spacy.lang.zh.examples import sentences 
nlp = spacy.load("zh_core_web_sm")
doc = nlp(sentences[0])
text = """\
字节对编码是一种简单的数据压缩形式，这种方法用数据中不存的一个字节表示最常出现的连续字节数据。\
这样的替换需要重建全部原始数据。字节对编码实例: 假设我们要编码数据 aaabdaaabac, 字节对“aa” \
出现次数最多，所以我们用数据中没有出现的字节“Z”替换“aa”得到替换表Z <- aa 数据转变为 ZabdZabac. \
在这个数据中，字节对“Za”出现的次数最多，我们用另外一个字节“Y”来替换它（这种情况下由于所有的“Z”都将被替换，\
所以也可以用“Z”来替换“Za”），得到替换表以及数据 Z <- aa, Y <- Za, YbdYbac.
"""
doc = nlp(text)
sentences = [sent.text for sent in doc.sents]
print(sentences)

In [ ]:
# A modern and fast NLP library that includes support for sentence segmentation. 
# spaCy uses a statistical model to predict sentence boundaries, which can be more accurate 
# than rule-based approaches for complex texts.

nlp = spacy.load("en_core_web_sm")
doc = nlp("Here is a sentence. Here is another one! And the last one.")
sentences = [sent.text for sent in doc.sents]
for ind, sent in enumerate(sentences):
    print(f"sentence-{ind}: {sent}\n")

- **Lemmatization (词形还原) and Stemming (词干提取)**

Lemmatization is the task of determining that two words have the same root, despite their surface differences. For some NLP situations, we also want two morphologically different forms of a word to behave similarly. For example in web search, someone may type the string woodchucks but a useful system might want to also return pages that mention woodchuck with no s.
  - **Example 1**: The words <span style="color:red;"><b>am</b></span>, are, and is have the shared lemma <span style="color:red;"><b>be</b></span>.
  - **Example 2**: The words <span style="color:red;"><b>dinner</b></span> and <span style="color:red;"><b>dinners</b></span> both have the lemma <span style="color:red;"><b>dinner</b></span>.

Lemmatization algorithms can be complex. For this reason we sometimes make use of a simpler but cruder method, which mainly consists of chopping off words final affixes. This naive version of morphological analysis is called stemming.

In [ ]:
import spacy
import pandas as pd

prompt = "Fudan University is located in Shanghai. are Universities"
nlp = spacy.load("en_core_web_sm")
doc = nlp(prompt)

df = pd.DataFrame([{
    "text": t.text,
    "lemma": t.lemma_, # Lemma: The base form of the word.
    "pos": t.pos_, # POS: The simple UPOS part-of-speech tag.
    "tag": t.tag_, # Tag: The detailed part-of-speech tag.
    "dep": t.dep_, # Dep: Syntactic dependency, i.e. the relation between tokens.
    "shape": t.shape_, # Shape: The word shape – capitalization, punctuation, digits.
    "is_alpha": t.is_alpha, # is alpha: Is the token an alpha character?
    "is_stop": t.is_stop, # is stop: Is the token part of a stop list.
} for t in doc])
df

<a id="30"></a>
## 3. Datasets Exploration

<a id="31"></a>
### 3.1 Heap\'s Law

Word types and word instances (tokens)

- **Word types** are the number of distinct words in a corpus; if the set of words in the vocabulary is
  $$
  V =\{v_1,v_2,\ldots,v_{|V|}\},
  $$
  where $|V|$ is the number of types. We also call it, the vocabulary size $|V|$. 

- **Word instances**: Given a text corpus, we treat it as a sequence of tokens $[w_1,w_2,\ldots,w_T]$ after tokenization. We call $T$, the total number $T$ of running words in this corpus.

- $T$: the number of word instances of corpus
- $|V|$: the number of word types

The larger the corpora we look at, the more word types we find, and in fact this relationship between $|V|$ and $T$ is called **Herdan's Law** or **[Heaps' Law](https://en.wikipedia.org/wiki/Heaps%27_law)** after its discoverers (in linguistics and information retrieval respectively). Given $k$ and $\beta$ positive constants, and $0<\beta<1$, it has the following form

$$
|V|=k \cdot T^\beta,
$$

the value of $\beta$ depends on the corpus size and the genre. With English text corpora, typically $k\in [10,100]$, and $\beta \in [0.4, 0.6]$. For the large corpora, $\beta$ ranges from .67 to .75. Roughly then we can say that the vocabulary size for a text goes up significantly faster than the square root of its length in words.  Let us test it!
Check more on 

<a id="32"></a>
### 3.2 Explore wikitext-2 dataset

- **Please note that huggingface cannot be directly used in China. An alternative way is to use [https://hf-mirror.com/](https://hf-mirror.com/).** If the below code does not work, please add followwing env variable in your terminal
  
  ```export HF_ENDPOINT=https://hf-mirror.com```

- **Explore wikitext-2 dataset**

In [ ]:
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

from datasets import load_dataset
# loads train/validation/test
wiki2_ds = load_dataset("wikitext", "wikitext-2-raw-v1")
wiki2_train = wiki2_ds["train"]

In [ ]:
summary = pd.DataFrame([
    {
        "split": name,
        "num_rows": d.num_rows,
        "num_columns": d.num_columns,
        "columns": ", ".join(d.column_names),
    }
    for name, d in wiki2_ds.items()
])
summary

In [ ]:
preview = []
for name, d in wiki2_ds.items():
    for i in range(5):
        preview.append({"split": name, "idx": i, **d[i]})

pd.DataFrame(preview)

- **Simple tokenization and testing the Heap's law.**

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

start_time = time.time()
wiki2_train = wiki2_ds["train"]
word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?") # simple word tokenizer

def heaps_curve(dataset, step=1000):
    V = set()
    N = 0
    Ns, Vs = [], []
    for ex in dataset:
        text = ex["text"]
        if not text or text.strip() == "": # empty word -> continue
            continue
        words = word_re.findall(text.lower()) # make all words to low cases
        for w in words:
            N += 1
            V.add(w)
            if N % step == 0:
                Ns.append(N)
                Vs.append(len(V))
    return np.array(Ns), np.array(Vs)

Ns, Vs = heaps_curve(wiki2_train, step=1000)

# Fit log |V| = log k + beta log N  -> linear regression on logs
logN = np.log(Ns)
logV = np.log(Vs)
beta, logk = np.polyfit(logN, logV, 1)
k = np.exp(logk)

print(f"Fitted Heaps' law: |V| ≈ {k:.2f} * T^{beta:.3f}")
print(f"Total runtime: {time.time() - start_time:.3f} seconds")

# Plot (log-log)
plt.figure(figsize=(7, 5))
plt.loglog(Ns, Vs, marker='o', linestyle='none', markersize=5, label="Empirical (Wikitext-2)")

# fitted line
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
Vs_line = k * (Ns_line ** beta)
plt.loglog(Ns_line, Vs_line, label=fr"Fitted ($k={k:.3f},\beta=${beta:.3f})")

plt.xlabel("$T$ (tokens)", fontsize = 15)
plt.ylabel("$|V|$ (vocab)", fontsize = 15)
plt.title(r"Heaps' Law (Simple tokenization): $|V|=k \cdot T^{\beta}$", fontsize = 15)
plt.legend(fontsize = 15)
plt.tight_layout()
plt.show()

- **Use spaCy tokenization and test Heap's Law**

In [ ]:
start_time = time.time()

nlp = spacy.load(
    "en_core_web_sm", 
    disable=["tagger","parser","ner","lemmatizer"]
)
wiki2_train = wiki2_ds["train"]

def heaps_curve_spacy(dataset, step=10_000, batch_size=256):
    V = set()
    N = 0
    Ns, Vs = [], []

    texts = (ex["text"] for ex in dataset if ex["text"] and ex["text"].strip())
    for doc in nlp.pipe(texts, batch_size=batch_size):
        for tok in doc:
            if tok.is_alpha: # choose your definition of "word"
                w = tok.text.lower()
                N += 1
                V.add(w)
                if N % step == 0:
                    Ns.append(N)
                    Vs.append(len(V))
    return np.array(Ns), np.array(Vs)

Ns, Vs = heaps_curve_spacy(wiki2_train, step=1000)

# Fit log |V| = log k + beta log N
logN = np.log(Ns)
logV = np.log(Vs)
beta, logk = np.polyfit(logN, logV, 1)
k = np.exp(logk)

print(f"Fitted Heaps' law (spaCy): |V| ≈ {k:.2f} * T^{beta:.3f}")
print(f"Total runtime: {time.time() - start_time:.3f} seconds")
# Plot (log-log)
plt.figure(figsize=(7, 5))
plt.loglog(Ns, Vs, marker='o', linestyle='none', markersize=5, label="Empirical (Wikitext-2)")

# fitted line
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
Vs_line = k * (Ns_line ** beta)
plt.loglog(Ns_line, Vs_line, label=fr"Fitted ($k={k:.3f},\beta=${beta:.3f})")

plt.xlabel("$T$ (tokens)", fontsize = 15)
plt.ylabel("$|V|$ (vocab)", fontsize = 15)
plt.title(r"Heaps' Law (Spacy tokenization): $|V|=k\cdot T^{\beta}$", fontsize = 15)
plt.legend(fontsize = 15)
plt.tight_layout()
plt.show()

<a id="33"></a>
### 3.3 Explore wikitext-103 dataset


We can download a larger dataset from the [hf-mirror](https://hf-mirror.com/):

```shell
export HF_HOME=$HOME/.cache/huggingface

huggingface-cli download Salesforce/wikitext --repo-type dataset --resume-download --include "wikitext-103-raw-v1/*.parquet"
```

- **Explore wikitext-103**

In [ ]:
from datasets import load_dataset
wiki103_ds = load_dataset("wikitext", "wikitext-103-raw-v1")  # will hit cache if present

In [ ]:
summary = pd.DataFrame([
    {
        "split": name,
        "num_rows": d.num_rows,
        "num_columns": d.num_columns,
        "columns": ", ".join(d.column_names),
    }
    for name, d in wiki103_ds.items()
])
summary

In [ ]:
start_time = time.time()
wiki103_train = wiki103_ds["train"]
word_re = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def heaps_curve(dataset, step=1000):
    V = set()
    N = 0
    Ns, Vs = [], []
    for ex in dataset:
        text = ex["text"]
        if not text or text.strip() == "": # empty word -> continue
            continue
        words = word_re.findall(text.lower()) # make all words to low cases
        for w in words:
            N += 1
            V.add(w)
            if N % step == 0:
                Ns.append(N)
                Vs.append(len(V))
    return np.array(Ns), np.array(Vs)

Ns, Vs = heaps_curve(wiki103_train, step=1000)

# Fit log |V| = log k + beta log N  -> linear regression on logs
logN = np.log(Ns)
logV = np.log(Vs)
beta, logk = np.polyfit(logN, logV, 1)
k = np.exp(logk)

print(f"Fitted Heaps' law (spaCy): |V| ≈ {k:.2f} * T^{beta:.3f}")
print(f"Total runtime: {time.time() - start_time:.3f} seconds")
# Plot (log-log)
plt.figure(figsize=(7, 5))
plt.loglog(Ns, Vs, marker='o', linestyle='none', markersize=5, label="Empirical (Wikitext-103)")

# fitted line
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
Vs_line = k * (Ns_line ** beta)
plt.loglog(Ns_line, Vs_line, label=fr"Fitted ($k={k:.3f},\beta=${beta:.3f})")

plt.xlabel("$N$ (tokens)", fontsize = 15)
plt.ylabel("$|V|$ (vocab)", fontsize = 15)
plt.title(r"Heaps' Law (Spacy tokenization): $|V|=k \cdot T^{\beta}$", fontsize = 15)
plt.legend(fontsize = 15)
plt.tight_layout()
plt.show()

<a id="34"></a>
### 3.4 BookCorpus datasets (GPT-1)

BookCorpus dataset has been used in training GPT-1. See <a href="/files/papers/GPT1-Improving_Language_Understanding_by_Generative_Pre-Training-2018.pdf"
   target="_blank" rel="noopener">
  GPT-1 paper (PDF)
</a>

- **Explore the BookCorpus datasets**

You can create *.dataset* (note it is .dataset not dataset) folder in llm-26 and download our file from [url](https://pan.baidu.com/s/115KyzS6sLCQjfgx9Ihh7nw?).

In [ ]:
import time
import tarfile

path = "../.datasets/books1.tar.gz"
start_time = time.time()
with tarfile.open(path, "r:gz") as tar:
    names = tar.getnames()
    print("num members:", len(names))
    print("\n".join(names[:20]))
print(f"total time to scan BookCorpus: {time.time() - start_time:.2f} seconds")

You can check ```tokenization_bookcorpus.py``` to generate heaps statistics in jsonl file. (We generated them files already.)

- **Heap's Law of BookCorpus dataset**

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def load_heaps_jsonl(log_path: str):
    Ns, Vs = [], []
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            Ns.append(r["total_tokens"])
            Vs.append(r["vocab_size"])
    return np.array(Ns), np.array(Vs)

Ns, Vs = load_heaps_jsonl("assets/books1_wordcounts.heaps_log.jsonl")

# fit log |V| = log k + beta log N
beta, logk = np.polyfit(np.log(Ns), np.log(Vs), 1)
k = np.exp(logk)
print(f"Fit: |V| ≈ {k:.2f} * T^{beta:.3f}")


plt.figure(figsize=(7,5))

plt.loglog(Ns, Vs, marker="o", linestyle="none", 
           markersize=5, label="Empirical (BookCorpus)")
Ns_line = np.linspace(Ns[0], Ns[-1], 200)
plt.loglog(Ns_line, k * (Ns_line ** beta), label=fr"Fitted Curve $(k={k:.3f},\beta=${beta:.3f})")
plt.xlabel("$T$ (tokens)", fontsize = 14)
plt.ylabel("$|V|$ (vocab)", fontsize = 14)
plt.title(r"Heaps' Law (Simple tokenization): $|V|=k \cdot T^{\beta}$", fontsize = 14)
plt.legend(fontsize = 14)
plt.tight_layout()
plt.show()

<a id="40"></a>
## 4. LLMs tokenization (Modern Way)

In general, there are two type of tokenizations

- **Top-down tokenization**: It defines a standard and implement rules to implement that kind of tokenization. It includes, for example, word tokenization, charater tokenization, and others.
  
- **Bottom-up tokenization**: It uses simple statistics of letter sequences to break up words into subword tokens. Subword tokenization (modern LLMs use this type!) belongs to this type.

In case of bottom-up tokenization, there are three standard way to do modern LLM tokenization:

- **BPE tokenization:** <a href="/files/papers/BPE-A_New_Algorithm_for_Data_Compression-1994.pdf"
   target="_blank" rel="noopener">[Original BPE paper]</a>, <a href="/files/papers/GPT2-Language_Models_are_Unsupervised_Multitask_Learners-2019.pdf"
   target="_blank" rel="noopener">[Adopted in GPT-2]</a>
  
- **Wordpiece tokenization:** <a href="/files/papers/Wordpiece-Japanese_and_Korean_voice_search-1994.pdf"
   target="_blank" rel="noopener">[Original Wordpiece paper]</a>, <a href="/files/papers/GNMT-Google_Neural_Machine_Translation_System_Bridging_the_Gap_between_Human_and_Machine_Translation-2016.pdf"
   target="_blank" rel="noopener">[Adopted in GNMT]</a>, <a href="/files/papers/BERT-Pre-training_of_Deep_Bidirectional_Transformers_for_Language_Understanding-2018.pdf"
   target="_blank" rel="noopener">[Adopted in BERT]</a>

- **Unigram tokenization:** <a href="/files/papers/Unigram-Subword_Regularization_Improving_Neural_Network_Translation_Models_with_Multiple_Subword_Candidates-2018.pdf"
   target="_blank" rel="noopener">[Original Unigram paper]</a>, <a href="/files/papers/ALBERT-A_Lite_BERT_for_Self-supervised_Learning_of_Language_Representations-2020.pdf"
   target="_blank" rel="noopener">[Adopted in AlBERT]</a>, <a href="/files/papers/T5-model-Exploring_the_Limits_of_Transfer_Learning_with_a_Unified_Text-to-Text_Transformer-2020.pdf"
   target="_blank" rel="noopener">[Adopted in T5]</a>

In [ ]:
import tiktoken

<a id="41"></a>
### 4.1 BPE tokenization

- **Explore trained tokenizer: [tiktoken](https://github.com/openai/tiktoken)** Any trained tokenizer has two functions:
  - **encode**: it maps running text into token ids (integers)
  - **decode**: it maps token id back to each text token (strings)

- It has been adopted from all modern LLMs including ChatGPT, GPT-series, and many others.
- tiktoken uses a byte-level BPE. (The emoji like 🤗 could be encoded as multiple UTF-8 bytes.)

#### 4.1.1 explore tiktoken tokenizer

The [tiktoken tokenizer](https://github.com/openai/tiktoken).

In [ ]:
# Load cl100k_tokenizer (it was used by gpt-3.5-turbo family, gpt-4, and also gpt-4-turbo

cl100k_tokenizer = tiktoken.get_encoding("cl100k_base")

text = "Don't you love 🤗 Transformers? We sure do."
token_ids = cl100k_tokenizer.encode(text)
token_strs = [cl100k_tokenizer.decode([tid]) for tid in token_ids]

print("ids:", token_ids)
print("tokens:", token_strs)

- what you’re seeing (' �', '�', '�') is normal. Those are individual bytes / partial UTF-8 pieces that don’t form a valid standalone Unicode character when decoded one token at a time.
- Next, we explore o200k_base, which is the tokenizer (encoding) used by OpenAI’s newer “o-series” models, especially the GPT-4o family. If you call a model like gpt-4o / gpt-4o-mini, the tokenization is o200k_base.

In [ ]:
gpt35_tokenizer = tiktoken.get_encoding("o200k_base")

token_ids = gpt35_tokenizer.encode(text)
token_strs = [gpt35_tokenizer.decode([tid]) for tid in token_ids]

print("ids:", token_ids)
print("tokens:", token_strs)

# Count tokens by counting the length of the list returned by .encode().
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens
print("---")

text = "Don't you love 🤗 Transformers? We sure do."
print(f'\"{text}\" has been encoded into {num_tokens_from_string(text, "cl100k_base")} subwords')
print(f'\"{text}\" has been encoded into {num_tokens_from_string(text, "o200k_base")} subwords')

- Note the tokens generated above is slightly different from ones generated by cl100k_base.

In [ ]:
# .decode() converts a list of token integers to a string.
encode_ids = [83, 1609, 5963, 374, 2294, 0, 11, 2025]
print(f'the decoded string is: \"{gpt35_tokenizer.decode(encode_ids)}\"')

- Try a slightly long text.

In [ ]:
text = """
Chapters 5 to 8 teach the basics of 🤗 Datasets and 🤗 Tokenizers before diving into classic NLP tasks.\
By the end of this part, you will be able to tackle the most common NLP problems by yourself. \
By the end of this part, you will be ready to apply 🤗 Transformers to (almost) any machine \
learning problem! E=mc^2. f(x) = x^2+y^2, print('hello world!’) baojianzhou. asdasfasdgasdg
"""
token_ids = gpt35_tokenizer.encode(text)
token_strs = [gpt35_tokenizer.decode([tid]) for tid in token_ids]
print(token_ids)
print(token_strs)

- **Limitations of general tokenizers**

**Generic tokenizers can split domain strings (genes/SMILES) in semantically harmful ways; we often need domain-specific tokenization (k-mers / specialized vocabularies) or character-level modeling.** Let us consider the following example

In [ ]:
text = """acetylsalicylic acid, aspirin, Group of substances: organic Physical appearance: \
colorless needles crystals Empirical formula (Hill's system for organic substances): \
C9H8O4 Structural formula as text: CH3COOC6H4COOH, 
the smiles format of it is CC(=O)OC1=CC=CC=C1C(O)=O."""

token_ids = gpt35_tokenizer.encode(text)
token_strs = [gpt35_tokenizer.decode([tid]) for tid in token_ids]
print(token_ids)
print(token_strs) # CC(=O)OC1=CC=CC=C1C(O)=O. should be a whole, but it is tokenized into small pieces.

#### 4.1.2 train a BPE tokenizer using tokenizers

- **tokenizers is fast**
  
  It is extremely fast (both training and tokenization), thanks to the Rust implementation. Takes less than 20 seconds to tokenize a GB of text on a server's CPU.
  
- **Example of training BPE using tokenizers**

  We will use wikitext-103 to train a tokenizer
    - Start with all the characters present in the training corpus as tokens.
    - Identify the most common pair of tokens and merge it into one token.
    - Repeat until the vocabulary (e.g., the number of tokens) has reached the size we want.

- **Step 1: pretraining for tokenization learner (adding whitespace)**

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# load wikitext-103
wiki103_ds = load_dataset("wikitext", "wikitext-103-raw-v1")  # train/validation/test

# Step 1: pre-training
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

- **Step 2: training (using BPE algorithm)**

In [ ]:
start_time = time.time()

# build tokenizer + trainer
trainer = BpeTrainer(
    vocab_size=30000,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]", "[MASK]", "[CLS]", "[SEP]"]
)
# iterator over ALL splits
def batch_iterator(batch_size=1000):
    for split in ["train", "validation", "test"]:
        for i in range(0, len(wiki103_ds[split]), batch_size):
            yield ds[split][i:i+batch_size]["text"]
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
# It will take above 1 second
print(f"Total runtime of training tokenizer on wikitext-103: {time.time() - start_time:.3f} seconds")

- **Step 3: save the tokenizer**

In [ ]:
# Step 3: save the trained tokenizer
tokenizer.save("assets/wikitext103-bpe.json")

In [ ]:
tokenizer = Tokenizer.from_file("assets/wikitext103-bpe.json")
output = tokenizer.encode("Hello, y'all! How are you 😁 ?")
print(output.tokens)
print(output.ids)

<a id="42"></a>
### 4.2 WordPiece tokenization

- **Step 1: normalization**

In [ ]:
# Step 1: normalization, Normalization is, in a nutshell, a set of operations you apply to 
# a raw string to make it less random or “cleaner”. Common operations include stripping 
# whitespace, removing accented characters or lowercasing all text. 

from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents
normalizer = normalizers.Sequence([NFD(), StripAccents()])

print(normalizer.normalize_str("Héllò hôw are ü?"))
tokenizer.normalizer = normalizer

- **Step 2: Pre-Tokenization**

In [ ]:
# Step 2: Pre-Tokenization, Pre-tokenization is the act of splitting a text into smaller 
# objects that give an upper bound to what your tokens will be at the end of training. 
# A good way to think of this is that the pre-tokenizer will split your text into “words”
# and then, your final tokens will be parts of those words. An easy way to pre-tokenize 
# inputs is to split on spaces and punctuations, which is done by the pre_tokenizers.
from tokenizers.pre_tokenizers import Whitespace
pre_tokenizer = Whitespace()
pre_tokenizer.pre_tokenize_str("Hello! How are you? I'm fine, thank you.")

In [ ]:
from tokenizers import pre_tokenizers
from tokenizers.pre_tokenizers import Digits
pre_tokenizer = pre_tokenizers.Sequence([Whitespace(), Digits(individual_digits=True)])
pre_tokenizer.pre_tokenize_str("Call 911!")

- **Step 3: Choose your model (WordPiece)**

Choose your tokenization algorithm. 

- models.BPE
- models.Unigram
- models.WordLevel
- models.WordPiece

- **Step 4: Post-Processing**

Post-processing is the last step of the tokenization pipeline, to perform any additional transformation to the Encoding before it’s returned, like adding potential special tokens.

In [ ]:
from tokenizers.processors import TemplateProcessing
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", 1), ("[SEP]", 2)],
)

In [ ]:
# BERT tokenization building

start_time = time.time()

from transformers import AutoTokenizer
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
bert_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

from tokenizers import normalizers
from tokenizers.normalizers import NFD, Lowercase, StripAccents
bert_tokenizer.normalizer = normalizers.Sequence([NFD(), Lowercase(), StripAccents()])

from tokenizers.pre_tokenizers import Whitespace
bert_tokenizer.pre_tokenizer = Whitespace()

from tokenizers.processors import TemplateProcessing
bert_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", 1),
        ("[SEP]", 2),
    ],
)
from tokenizers.trainers import WordPieceTrainer
trainer = WordPieceTrainer(vocab_size=30522, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

# iterator over ALL splits
def batch_iterator(batch_size=1000):
    for split in ["train", "validation", "test"]:
        for i in range(0, len(wiki103_ds[split]), batch_size):
            yield ds[split][i:i+batch_size]["text"]
bert_tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
# It will take above 1 second
print(f"Total runtime of training tokenizer on wikitext-103: {time.time() - start_time:.3f} seconds")
bert_tokenizer.save("assets/wikitext103-bert.json")

- **Step 5: Encoding and Decoding**

In [ ]:
output = bert_tokenizer.encode("Hello, y'all! How are you 😁 ?")
print(output.ids)
# [1, 5993, 919, 16, 67, 11, 1772, 5, 1972, 1700, 2358, 0, 35, 2]
bert_tokenizer.decode([1, 5993, 919, 16, 67, 11, 1772, 5, 1972, 1700, 2358, 0, 35, 2])
# "Hello , y ' all ! How are you ?"

In [ ]:
output = bert_tokenizer.encode("Welcome to the 🤗 Tokenizers library.")
print(output.tokens)
# ["[CLS]", "welcome", "to", "the", "[UNK]", "tok", "##eni", "##zer", "##s", "library", ".", "[SEP]"]
bert_tokenizer.decode(output.ids)
# "welcome to the tok ##eni ##zer ##s library ."

<a id="43"></a>
### 4.3 Unigram tokenization

<a id="44"></a>
### 4.4 Huggingface pretrained tokenizers

PreTrainedTokenizer, PreTrainedTokenizerBase, AutoTokenizer

- https://github.com/huggingface/transformers/blob/main/src/transformers/tokenization_utils.py
- https://github.com/huggingface/transformers/blob/main/src/transformers/tokenization_utils_base.py
- https://github.com/huggingface/transformers/blob/main/src/transformers/tokenization_utils_fast.py
- Check tokenizers at https://github.com/huggingface/transformers/blob/main/setup.py
- If you want to train a tokenizer by yourself, then go to: https://github.com/huggingface/tokenizers
- A fast BPE tokenizer is also at: https://github.com/openai/tiktoken
- An implementation of sentencepiece is at: https://github.com/google/sentencepiece
- There are 3 most common methods for tokenization: https://github.com/huggingface/tokenizers/tree/main/tokenizers/src/models
- - BPE: https://aclanthology.org/P16-1162.pdf
  - Unigram: https://arxiv.org/pdf/1804.10959
  - WordPiece https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/37842.pdf

In [ ]:
from transformers import Qwen2Tokenizer, Qwen2TokenizerFast

In [ ]:
tokenizer = Qwen2TokenizerFast.from_pretrained(pretrained_model_name_or_path="Qwen/Qwen-tokenizer")
text = "What is natural language processing? 🤗"
encoded = tokenizer(text, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])

In [ ]:
from transformers import AutoTokenizer
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3")
# Example: tokenize some text
text = "What is natural language processing? 🤗"
encoded = tokenizer(text, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])
decoded = tokenizer.decode(encoded["input_ids"][0])
print("Decoded text:", decoded)
decoded = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True)
print("Decoded with special text:", decoded)

In [ ]:
vocab = tokenizer.get_vocab()
print("Vocab size (attribute):", tokenizer.vocab_size)
print("Vocab size (dict length):", len(vocab))
id_to_token = {id_: tok for tok, id_ in vocab.items()}
pairs = [[i, repr(id_to_token[i])]for i in range(20)] # show first 20 tokens
print(pairs)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-72B-Instruct")
text = "What is natural language processing? 🤗"
encoded = tokenizer(text, return_tensors="pt")
print("Input IDs:", encoded["input_ids"])
# Decode including special tokens
decoded = tokenizer.decode(encoded["input_ids"][0])
print("Decoded text:", decoded)

In [ ]:
from transformers import AutoTokenizer
# Download vocabulary from huggingface.co and cache.
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# Download vocabulary from huggingface.co (user-uploaded) and cache.
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-cased")
# If vocabulary files are in a directory (e.g. tokenizer was saved using *save_pretrained('./test/saved_model/')*)
# tokenizer = AutoTokenizer.from_pretrained("./test/bert_saved_model/")
# Download vocabulary from huggingface.co and define model-specific arguments
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base", add_prefix_space=True)

<a id="50"></a>
## 5. Minimum Edit Distance

<a id="51"></a>
### 5.1 Algorithm analysis

Assessing the similarity between two strings is essential in many NLP tasks. For spell correction, one must choose the most similar word to a misspelled input from a set of candidates. For example, if ```graffe"``` is the input, among the options ```graf```, ```graft```, ```grail```, and ```giraffe```, the task is to select the word closest to the input based on a certain criterion. Similarly, comparing a translated sentence to a reference translation helps evaluate the performance of translation models in machine translation. This process may involve calculating the minimum edit distance to determine the similarity of two sentences. This note gives a dynamic programming algorithm for MED.

<div style="background:#f5f5f5; border:1px solid #ddd; border-left:6px solid #777; border-radius:8px; padding:12px 16px; margin:12px 0;">
  <b>Definition (Minimum Edit Distance, MED).</b><br>
  Given two strings $X=[x_1,x_2,\ldots,x_n]$ and $Y=[y_1,y_2,\ldots,y_m]$, the minimum edit distance between $X$ and $Y$ is the minimum number of edit operations required to convert $X$ into $Y$.<br><br>
  <b>Allowed edit operations:</b>
  <ol style="margin: 6px 0 0 18px;">
    <li><b>Insertion:</b> add a character to $X$.</li>
    <li><b>Deletion:</b> remove a character from $X$.</li>
    <li><b>Substitution:</b> replace a character in $X$ with another character.</li>
  </ol>
  <div style="margin-top:8px;">
    These operations are applied sequentially from left to right to obtain $Y$.
  </div>
</div>

<div style="background:#f5f5f5; border:1px solid #ddd; border-left:6px solid #777; border-radius:8px; padding:12px 16px; margin:12px 0;">
  <b>Example.</b> Suppose $X=\texttt{INTENTION}$ and $Y=\texttt{EXECUTION}$. One possible process of transforming $X \rightarrow Y$ is:
</div>

**Step-by-step edits (one possible alignment):**

1. $\texttt{INTENTION} \;\xrightarrow{\text{Del}(\texttt{I})}\; \texttt{NTENTION}$
2. $\texttt{NTENTION} \;\xrightarrow{\text{Sub}(\texttt{N}\rightarrow\texttt{E})}\; \texttt{ETENTION}$
3. $\texttt{ETENTION} \;\xrightarrow{\text{Sub}(\texttt{T}\rightarrow\texttt{X})}\; \texttt{EXENTION}$
4. $\texttt{EXENTION} \;\xrightarrow{\text{Ins}(\texttt{C})}\; \texttt{EXECNTION}$
5. $\texttt{EXECNTION} \;\xrightarrow{\text{Sub}(\texttt{N}\rightarrow\texttt{U})}\; \texttt{EXECUTION} = Y$

We assume that **Ins** and **Del** each count as **1** operation, while **Sub** counts as **2** operations.  
Consequently, the total number of edit operations above is $1 + 2 + 2 + 1 + 2 = 8$.

Let $O_k=[o_1,o_2,\ldots,o_k]$ denote the sequence of operations transforming $X$ into $Y$, where each $o_i \in \{\text{Ins},\text{Del},\text{Sub}\}$.  
We call $O_k$ an <span style="color:#1f77b4; font-weight:600;">alignment</span> of $X$ into $Y$ (see the figure below).

<div style="text-align:center;">
  <img src="assets/med-example.svg" style="width:65%; max-width:600px;">
</div>

- **Dynamic Programming for MED**

A naive approach would examine all possible edit sequences $O_k$ and choose the one with the minimal cost. However, this method has **exponential** time complexity. Instead, we break the problem into smaller subproblems and exploit **optimal substructure**.

<div style="background:#f5f5f5; border:1px solid #ddd; border-left:6px solid #777; border-radius:8px; padding:12px 16px; margin:12px 0;">
  <b>Problem setting.</b> For $i=\{0,1,\ldots,n\}$ and $j=\{0,1,\ldots,m\}$, let
  <ul style="margin:8px 0 0 18px;">
    <li>$X_i = [x_1, x_2, \ldots, x_i]$ be the prefix of $X$ of length $i$.</li>
    <li>$Y_j = [y_1, y_2, \ldots, y_j]$ be the prefix of $Y$ of length $j$.</li>
  </ul>
  By default, $X_0=\emptyset$ and $Y_0=\emptyset$. Define $D[i,j]$ as the <b>minimum edit distance</b> from $X_i \rightarrow Y_j$.  Clearly, $D[n,m]$ is the MED of $X \rightarrow Y$. There are two base cases:
  <ul style="margin:8px 0 0 18px;">
    <li>$D[i,0]=i$ (delete all $i$ characters from $X_i$).</li>
    <li>$D[0,j]=j$ (insert all $j$ characters into $\emptyset$).</li>
  </ul>

  In the rest, we assume $i,j \ge 1$.
</div>

<div style="background:#f5f5f5; border:1px solid #ddd; border-left:6px solid #777; border-radius:8px; padding:12px 16px; margin:12px 0;">
  <b>Optimal substructure.</b> Let $O_k=[o_1,o_2,\ldots,o_k]$ be a minimum-cost sequence of operations that transforms $X_i \rightarrow Y_j$. Then
  $$
  X_i = X_i^{0} \xrightarrow{o_1} X_i^1 \xrightarrow{o_2} X_i^2 \rightarrow \cdots \xrightarrow{o_{k-1}} X_i^{k-1} \xrightarrow{o_k} X_i^k= Y_j .
  $$
  If we remove the last operation $o_k$, the remaining sequence $O_{k-1}=[o_1,o_2,\ldots,o_{k-1}]$ must be an optimal process for $X_i \rightarrow X_i^{k-1}$.
  Otherwise, suppose there exists another sequence $O'_{k-1}$ that transforms $X_i \rightarrow X_i^{k-1}$ with strictly smaller cost:
  $\operatorname{cost}(O'_{k-1}) < \operatorname{cost}(O_{k-1})$.
  Then combining $O'_{k-1}$ with the last operation $o_k$ would yield a sequence for $X_i \rightarrow Y_j$ with total cost strictly smaller than $\operatorname{cost}(O_k)$—a contradiction.
  This is the <span style="color:#1f77b4; font-weight:600;">optimal substructure</span> property.
</div>

Because operations proceed from the left of $X_i$ to the right, there are only **three** possibilities for the last operation $o_k$:

- **Case 1 (Deletion).** Delete $x_i$ from $X_i$.  
  $$
  D[i,j] = D[i-1,j] + \operatorname{Del}(x_i).
  $$

- **Case 2 (Insertion).** Insert $y_j$ into $Y_{j-1}$.  
  $$
  D[i,j] = D[i,j-1] + \operatorname{Ins}(y_j).
  $$

- **Case 3 (Substitution).** Substitute $x_i$ by $y_j$.  
  $$
  D[i,j] = D[i-1,j-1] + \operatorname{Sub}(x_i,y_j).
  $$

Therefore, MED for $X_i \rightarrow Y_j$ can be computed from the three subproblems
$D[i-1,j]$, $D[i,j-1]$, and $D[i-1,j-1]$ by taking the minimum:

$$
D[i,j] =
\min\begin{cases}
D[i-1,j] + \operatorname{Del}(x_i),\\[2pt]
D[i,j-1] + \operatorname{Ins}(y_j),\\[2pt]
D[i-1,j-1] + \operatorname{Sub}(x_i,y_j).
\end{cases}
$$

This recurrence is the core of **dynamic programming** for MED.   In the example above, we can compute the DP matrix $\mathbf{D}$ using this recurrence, as shown in the following tables.  The above formula is called dynamic procedure or dynamic programming. Given the above-mentioned example, we can calculate matrix $\mathbf{D}$ using this dynamic procedure as shown in the following tables.

- **DP Table: Initialization vs. Final Result**

<div style="display:flex; gap:24px; align-items:flex-start;">
  <div style="flex:1; min-width:0;">
    <div style="font-weight:800; margin-bottom:6px;">Initial stage: \(D[\cdot,\cdot]\)</div>
    $$ 
    % --- paste your FIRST table here ---
    \begin{array}{l|l|l|l|l|l|l|l|l|l|l}
    {N} & {9} &  &  &  &  &  &  &  &  &  \\
    \hline {O} & {8} &  &  &  &  &  &  &  &  &  \\
    \hline {I} & {7} &  &  &  &  &  &  &  &  &  \\
    \hline {T} & {6} &  &  &  &  &  &  &  &  &  \\
    \hline {N} & {5} &  &  &  &  &  &  &  &  &  \\
    \hline {E} & {4} &  &  &  &  &  &  &  &  &  \\
    \hline {T} & {3} &  &  &  &  &  &  &  &  &  \\
    \hline {N} & {2} &  &  &  &  &  &  &  &  &  \\
    \hline {I} & {1} &  &  &  &  &  &  &  &  &  \\
    \hline \# & {0} & {1} & {2} & {3} & {4} & {5} & {6} & {7} & {8} & {9} \\
    \hline & \# & {E} & {X} & {E} & {C} & {U} & {T} & {I} & {O} & {N}
    \end{array}
    $$</div>
    <div style="width:2px; align-self:stretch; background:rgba(0,0,0,0.18);"></div>
  <div style="flex:1; min-width:0;">
    <div style="font-weight:800; margin-bottom:6px;">Final stage: \(D[n,m]=8\)</div>
    $$
    % --- paste your SECOND table here ---
    \begin{array}{c|c|c|c|c|c|c|c|c|c|c}
     N & 9 & 8 & 9 & 10 & 11 & 12 & 11 & 10 & 9 & \color{red}{8} \\
    \hline O & 8 & 7 & 8 & 9 & 10 & 11 & 10 & 9 & 8 & 9 \\
    \hline I & 7 & 6 & 7 & 8 & 9 & 10 & 9 & 8 & 9 & 10 \\
    \hline T & 6 & 5 & 6 & 7 & 8 & 9 & 8 & 9 & 10 & 11 \\
    \hline N & 5 & 4 & 5 & 6 & 7 & 8 & 9 & 10 & 11 & 10 \\
    \hline E & 4 & 3 & 4 & 5 & 6 & 7 & 8 & 9 & 10 & 9 \\
    \hline T & 3 & 4 & 5 & 6 & 7 & 8 & 7 & 8 & 9 & 8 \\
    \hline N & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 7 & 8 & 7 \\
    \hline I & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 6 & 7 & 8 \\
    \hline \# & 0 & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 \\
    \hline & \# & E & X & E & C & U & T & I & O & N
    \end{array}
    $$
  </div>
</div>

<div style="background:#f5f5f5; border:1px solid #ddd; border-left:6px solid #777; border-radius:8px; padding:12px 16px; margin:12px 0;">
  <b>Remark.</b> We call the above method a <b>dynamic programming</b> method.
  <ol style="margin:8px 0 0 18px;">
    <li>Try to prove or disprove the <b>uniqueness</b> of the optimal sequence of operations.</li>
      <li>
  Can you find approximate solutions if $n$ and $m$ are large? (Hint: search for <a href="https://en.wikipedia.org/wiki/Approximate_string_matching" target="_blank" rel="noopener">approximate string matching</a>.)
</li>
  </ol>
</div>

<a id="52"></a>
### 5.2 Algorithm implementation

In [ ]:


# define minimum edit distance algorithm via dynamic programming
def minimum_edit_distance(source, target):
    n = len(source)
    m = len(target)
    d_mat = np.zeros((n + 1, m + 1))
    for i in range(1, n + 1):
        d_mat[i, 0] = i
    for j in range(1, m + 1):
        d_mat[0, j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = 0 if source[i - 1] == target[j - 1] else 2
            del_ = d_mat[i - 1][j] + 1
            ins_ = d_mat[i][j - 1] + 1
            d_mat[i][j] = min(del_, ins_, d_mat[i - 1][j - 1] + sub)
    trace, align_source, align_target = backtrack_alignment(source, target, d_mat)
    return d_mat[n, m], trace, align_source, align_target

def backtrack_alignment(source, target, d_mat):
    align_source, align_target = [], []
    i, j = len(source), len(target)
    back_trace = [[i, j]]

    while (i, j) != (0, 0):
        # boundary cases first (avoid negative indexing)
        if i == 0:
            back_trace.append([i, j - 1])
            align_source = ["*"] + align_source
            align_target = [target[j - 1]] + align_target
            j -= 1
            continue
        if j == 0:
            back_trace.append([i - 1, j])
            align_source = [source[i - 1]] + align_source
            align_target = ["*"] + align_target
            i -= 1
            continue

        sub_cost = 0 if source[i - 1] == target[j - 1] else 2

        # prefer substitution/match when optimal (your tie-break rule)
        if d_mat[i, j] == d_mat[i - 1, j - 1] + sub_cost:
            back_trace.append([i - 1, j - 1])
            align_source = [source[i - 1]] + align_source
            align_target = [target[j - 1]] + align_target
            i, j = i - 1, j - 1

        # deletion
        elif d_mat[i, j] == d_mat[i - 1, j] + 1:
            back_trace.append([i - 1, j])
            align_source = [source[i - 1]] + align_source
            align_target = ["*"] + align_target
            i -= 1

        # insertion
        else:
            back_trace.append([i, j - 1])
            align_source = ["*"] + align_source
            align_target = [target[j - 1]] + align_target
            j -= 1

    return back_trace, align_source, align_target

# test the minimum edit distance
def test_med(source, target):
    med, trace, align_source, align_target = minimum_edit_distance(source, target)
    print(f"input source: {source} and target: {target}")
    print(f"med: {med}")
    print(f"trace: {trace}")
    print(f"aligned source: {align_source}")
    print(f"aligned target: {align_target}")

In [ ]:
test_med(source="INTENTION", target="EXECUTION")

In [ ]:
test_med(source="AGGCTATCACCTGACCTCCAGGCCGATGCCC", target="TAGCTATCACGACCGCGGTCGATTTGCCCGAC")

<a id="60"></a>
## 6. Submit your work

In [ ]:
# Finally, you all done! Please submit your jupyter notebook via elearning!

your_chinese_name = "XYZ" # put your chinese name here
your_student_id = "" # put your student id here

jnk_end_time = time.time()
total = jnk_end_time - jnk_start_time
h, rem = divmod(int(total), 3600)
m, s = divmod(rem, 60)

print(f"Total runtime: {h} h {m} min {s} s")
print(f"I am {your_chinese_name}:{your_student_id}, I am doing a great job!")

- **Run the following command and submit the html version accordingly (it will produce an html file under the current folder)**

In [ ]:
!jupyter nbconvert lecture-01-exercise-tokenization.ipynb --to html --template classic --embed-images

<a id="70"></a>
## 7. Useful commands

- **1. Save your jupyter notebook as a PDF file**

To save your notebook as pdf you can use the following command

<div style="background:#f6f6f6; padding:6px 10px; border-radius:10px;"><pre style="margin:0; padding:0; background:transparent; line-height:1.2;"><code style="background:transparent; color:#000; padding:0; display:block;">jupyter nbconvert lecture-01-exercise-tokenization.ipynb --to html --template classic --embed-images</code></pre>
</div>

- **2. Save your env from installed history**

<div style="background:#f6f6f6; padding:6px 10px; border-radius:10px;"><pre style="margin:0; padding:0; background:transparent; line-height:1.2;"><code style="background:transparent; color:#000; padding:0; display:block;">conda env export --from-history > your-environment.yml</code></pre>
</div>

If you do not want to have prefix in the end of yml file, you can try to remove the prefix line by using:

<div style="background:#f6f6f6; padding:6px 10px; border-radius:10px;"><pre style="margin:0; padding:0; background:transparent; line-height:1.2;"><code style="background:transparent; color:#000; padding:0; display:block;">conda env export --from-history > your-environment.yml</br>sed -i '' '/^prefix: /d' environment.yml   # macOS sed</code></pre>
</div>

- **3. Start uvicorn server**

<div style="background:#f6f6f6; padding:6px 10px; border-radius:10px;"><pre style="margin:0; padding:0; background:transparent; line-height:1.2;"><code style="background:transparent; color:#000; padding:0; display:block;">uvicorn test_sd_server:app --host 127.0.0.1 --port 5055</code></pre>
</div>
